# E9 Multistage Training

Author: Arush Arora

## Introduction
This codebase has mostly consisted of additive Graph Positional Encodings (GREPs) injections to provide nodal embeddings to the LLMs at hand. This new multi-stage training will rely on training an R-PEARL/GT to simply replicate the graph before expecting it to serve the LLM with **multiplicative** GREPs, which will be factored directly into the attention-mask matrix for rendition to the LLM (as a Hadamard product cover on the attention logits). Thus, the system will be more carefully trained to incorporate the variation in model architecture among GNNs and LLMs (in terms of their pre-trained weights rather than simply their mathematical foundations).

## Mathematical Overview

## The R-PEARL GNN

The Random Positional Encoding (R-PEARL) GNN architecture is a PE generator that inputs white noise and processes it over an undirected graph $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathcal{W})$. In this work, the graph is represented by an adjacency matrix $A$, and the GNN composes [Topology Adaptive Graph (TAG)](https://arxiv.org/abs/1710.10370) Convolutional Layers with pointwise nonlinearities (demodulators).

### Graph Convolutional Network (GNN)
The code below establishes this project's implementation of a Graph Convolutional Network, which is the foundational architecture comprising R-PEARL. The equation to demonstrate the internal architecture of this NN as follows (in most cases, $\mathbf{P}(\cdot) = \mathbf{I}(\cdot)$, where $\mathbf{I}$ is the identity function):
$$\Phi(\mathbf{X}, \mathbf{S}, \mathcal{H}) = \mathbf{X}^{(L)}$$
$$\mathbf{X}^{(0)} = \mathbf{X} \qquad \mathbf{X}^{(l)} = \mathbf{P}\Bigg[\sigma\Bigg(\sum_{k = 0}^{K^{(l)} - 1} \mathbf{S}^k\mathbf{X}^{(l - 1)}\mathbf{H}_k^{(l)}\Bigg)\Bigg]$$

#### Transformer

The Transformer architecture follows that of the Llama3.1-8B distilled PRISM model. First, the TXT file, containing the scene-graph data, is tokenized and embedded into matrices $E$ and $\tilde{X}$ as follows, where $V$ is the size of the vocabulary and $d$ is the embedding dimension.

$$\text{TXT Tokenized Data from GPT-4: } E = \begin{bmatrix}
\mathbf{e}_1 & \mathbf{e}_2 & \overset{\mathbf{e}_t}{\cdots} & \mathbf{e}_T
\end{bmatrix}^\top \qquad \mathbf{e}_t \in \mathbb{R}^V$$

$$\text{Embed: } X = \begin{bmatrix}
\mathbf{x}_1 & \mathbf{x}_2 & \overset{\mathbf{x}_t}{\cdots} & \mathbf{x}_T
\end{bmatrix}^\top \qquad \mathbf{x}_t \in \mathbb{R}^d$$

Next, the transformer operates using the equations below:

$$X = \tilde{X} + P$$

$${Z}_{1:t}^{(L)} = \operatorname{Trf}\bigg({X}_{1:t}, {\mathcal{T}}_l\bigg) \qquad {\mathcal{T}}_l = \begin{bmatrix}
{Q}_l & {K}_l & {V}_l & \left({W}_o\right)_l
\end{bmatrix}^\top \in \mathbb{R}^{4 \times T \times D}$$

$$\hat{\mathbf{Y}}_{t + 1} = \operatorname{Linear}\Big({Z}_{1:t}^{(L)}\Big) \in \mathbb{R}^V$$
$$\text{Cross-Entropy Loss: } \mathcal{L}(E, \hat{\mathbf{Y}}) = \sum_t \sum_v e_{vt}\log{\hat{y}_t}$$

## Graph-Augmented LLM

The last class that is needed to create the full GREP-PRISM architecture is the `GraphAugmentedLLM`, which simply implements the following equation as a Neural Network object in PyTorch's `torch.nn` module (referring to above equations for definitions).
$\renewcommand{\utilde}[1]{\underset{\sim}{#1}}$ $$\utilde{P} = \hat{\mathbb{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \Phi\Big(\mathbf{q}^{(m)}, \utilde{S}, \mathcal{H}\Big)$$

$$\utilde{X} = \utilde{\tilde{X}} + \utilde{P}$$

$${\utilde{Z}}_{1:t}^{(L)} = \operatorname{Trf}\bigg({\utilde{X}}_{1:t}\bigg)$$

In [1]:
%env CUDA_VISIBLE_DEVICES=1

env: CUDA_VISIBLE_DEVICES=1


In [2]:
# Import modules.
import torch
import random
import networkx as nx
import matplotlib.pyplot as plt

from prism.data import data, compact_prompt, utils
from prism.models import inference, gnn_llm
from prism.eval import evaluate, loading

In [3]:
# Standard options.
eval_path = '../data_store/revised/gen/nav100_n30_gemma_data/split/test_graphs'
include_edge_list = False

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Setup the Gemma 4 model from Hugging Face.
llm = AutoModelForCausalLM.from_pretrained("google/gemma-4-12B-it", dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("google/gemma-4-12B-it")

# Initialize a barebones planner for testing.
model = gnn_llm.GraphMaskLLM(llm, use_edges=include_edge_list)
planner = inference.GraphAugmentedInMemoryLLM(model, tokenizer, include_edge_list)

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [5]:
from datasets import load_dataset

# Load in training dataset.
full_dataset = load_dataset("json", data_files=["../data/gen/nav100_n10_gemma_data/split/formatted_all_new_2turn__train.json"], split="train")
full_dataset = data.preprocess_dataset(
    full_dataset, tokenizer,
    architecture="graph_mask_llm",
    text_edge_list=include_edge_list,
)

In [6]:
# Setup eval infrastructure.
samples_by_graph, graph_file_by_name = loading.load_samples_by_graph(eval_path)
graph_file = random.choice(list(samples_by_graph.keys()))
eval_data = samples_by_graph[graph_file]
eval_data = {graph_file: [eval_data[random.randint(0, len(eval_data))]]}
eval_data

{'data_gen_004': [EvalSample(task='From the starting area, give the route to the area containing the microscope and list the path.', answer='(?i)\\bcommand_deck_1\\b.*\\bbio_lab_1\\b', graph={'objects': [{'name': 'transmitter_1', 'coords': [70.2, 5.3], 'description': ''}, {'name': 'mainframe_1', 'coords': [58.1, -11.7], 'description': ''}, {'name': 'microscope_1', 'coords': [-39.6, 31.0], 'description': ''}, {'name': 'beaker_1', 'coords': [-26.2, 39.4], 'description': ''}, {'name': 'hydro_pod_1', 'coords': [-30.1, 54.6], 'description': ''}, {'name': 'stasis_pod_1', 'coords': [-19.2, 54.8], 'description': ''}, {'name': 'wrench_1', 'coords': [-33.4, -49.7], 'description': ''}, {'name': 'fuel_rod_1', 'coords': [-51.2, -33.7], 'description': 'leaking'}, {'name': 'valve_1', 'coords': [-40.1, -46.3], 'description': ''}], 'regions': [{'name': 'command_deck_1', 'coords': [72.5, 19.9], 'description': ''}, {'name': 'bridge_1', 'coords': [59.5, -2.4], 'description': ''}, {'name': 'comms_hub_1', '

In [7]:
print(f'/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/graph_file.html')

/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/graph_file.html


## Experiments

### §1 Testing a Raw `GraphMaskLLM`

In [8]:
results = evaluate.eval_model_multiple_graphs(
    model, tokenizer, eval_data,
    include_edge_list=include_edge_list,
    use_icl=False,
    permutation=None,
    on_graph_done=None
)

[spine-llm] client=GraphAugmentedInMemoryLLM, prompt_tokens=495
[spine-llm] graph_found=True, n_graphs=1, robot_location=command_deck_1
[spine-llm] injection scope_start=255 / 495 tokens
[spine-llm] raw_output (first 500 chars): <think>
Relevant graph: command_deck_1, bridge_1, comms_hub_1, observation_dome_1, officer_quarters_1, navigation_bay_1, server_room_1, power_core_1, bio_lab_1, research_hall_1, sample_vault_1, chem_lab_1, data_archive_1, botany_wing_1, specimen_tank_1, cryo_chamber_1, engine_room_1, maintenance_tunnel_1, ballast_control_1, cargo_bay_1, airlock_1, reactor_pit_1, pump_station_1, waste_treatment_1, microscope_1.

Reasoning:
1. The robot starts at command_deck_1.
2. The target object is microscope_
{'primary_goal': '', 'relevant_graph': 'command_deck_1, bridge_1, comms_hub_1, observation_dome_1, officer_quarters_1, navigation_bay_1, server_room_1, power_core_1, bio_lab_1, research_hall_1, sample_vault_1, chem_lab_1, data_archive_1, botany_wing_1, specimen_tank_1, 